In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import classification_report
import joblib
from pathlib import Path
from gene_thesaurus import GeneThesaurus


In [2]:
MODELS_PATH = "/home/mariya/Data/pallas/models/gex"
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "cal-and-test-gex-1121")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.gex.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")
random_seed = 42
rf_model_path = os.path.join(MODELS_PATH, "rf_model.pkl")
selected_features_path = os.path.join(MODELS_PATH, "selected_features.txt")
pipeline_path = os.path.join(MODELS_PATH, "pipeline.pkl")
hyperparams_path = os.path.join(MODELS_PATH, "hyperparams.pkl")

base_dir = Path.cwd()
selected_features_list_path = base_dir / "output" / "selected_features.3.txt"

In [3]:
# Read in selected features as list
with open(selected_features_list_path, "r") as f:
    selected_features = [line.strip() for line in f.readlines()]

In [4]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_test, y_test

(             ENSG00000184507  ENSG00000079102  ENSG00000248329  \
 id                                                               
 SJHBBBM4PX3        -1.009419        -1.139177        -1.139177   
 SJHBBBM5PY5        -1.139177        -1.139177        -1.139177   
 SJHBBBM5TXY        -1.052304        -1.139177        -0.678568   
 SJHBBBM67H6        -1.062484        -1.139177        -1.139177   
 SJHBBBN9RDM        -1.032888        -1.139177        -1.139177   
 ...                      ...              ...              ...   
 16-335             -1.038952        -1.139177        -1.139177   
 16-338             -0.877979        -1.121756        -1.139177   
 16-343             -1.066091        -1.139177        -1.139177   
 16-346              6.701331        -1.006708        -1.139177   
 16-350              0.389281         0.440512         0.269102   
 
              ENSG00000101115  ENSG00000172752  ENSG00000097007  \
 id                                                         

In [5]:
# Get just selected features
X_test = X_test[selected_features]
X_test.shape

(1369, 196)

In [6]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test, y_test

(             ENSG00000184507  ENSG00000079102  ENSG00000248329  \
 id                                                               
 SJHBBBM4PX3        -1.009419        -1.139177        -1.139177   
 SJHBBBM5PY5        -1.139177        -1.139177        -1.139177   
 SJHBBBM5TXY        -1.052304        -1.139177        -0.678568   
 SJHBBBM67H6        -1.062484        -1.139177        -1.139177   
 SJHBBBN9RDM        -1.032888        -1.139177        -1.139177   
 ...                      ...              ...              ...   
 16-335             -1.038952        -1.139177        -1.139177   
 16-338             -0.877979        -1.121756        -1.139177   
 16-343             -1.066091        -1.139177        -1.139177   
 16-346              6.701331        -1.006708        -1.139177   
 16-350              0.389281         0.440512         0.269102   
 
              ENSG00000101115  ENSG00000145794  ENSG00000105642  \
 id                                                         

In [7]:
print("\ny_test class distribution:")
print(y_test.value_counts())


y_test class distribution:
class                   
B-ALL, high hyperdiploid    167
AML, KMT2A-r                154
B-ALL, ETV6::RUNX1          139
B-ALL, BCR::ABL1-like       115
AML, RUNX1::RUNX1T1         108
AML, NPM1alt                 68
B-ALL, KMT2A-r               66
B-ALL, PAX5alt               66
T-ALL                        53
B-ALL, DUX4-r                53
AML, CBFB::MYH11             50
B-ALL, BCR::ABL1             42
B-ALL, TCF3::PBX1            38
B-ALL, hypodiploid           37
AML, CEBPAalt                34
AML, NUP98-r                 29
B-ALL, ZNF384-r              25
B-ALL, ETV6::RUNX1-like      21
B-ALL, MEF2D-r               17
B-ALL, iAMP21                15
AML, DEK::NUP214             13
AML, CBFA2T3::GLIS2          11
B-ALL, PAX5 P80R             11
Non-leukemic                  7
AML, APL-PML::RARA            6
B-ALL, NUTM1-r                5
B-ALL, MYC-r                  5
AML, KAT6A-r                  4
B-ALL, HLF-r                  3
AML, ETV6-r        

In [8]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_test

array(['B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', ..., 'B-ALL, BCR::ABL1-like',
       'B-ALL, NUTM1-r', 'B-ALL, ETV6::RUNX1'],
      shape=(1369,), dtype=object)

In [9]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features from text file
selected_features_path = pd.read_csv(selected_features_list_path, header=None)[0].tolist()

# Use selected features for test set
y_pred_final = final_rf.predict(X_test)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test.shape}")

Number of selected features: 196
Shape of X_test_selected: (1369, 196)


In [10]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     0.833     0.909         6
     AML, CBFA2T3::GLIS2      1.000     0.909     0.952        11
        AML, CBFB::MYH11      1.000     0.980     0.990        50
           AML, CEBPAalt      0.971     0.971     0.971        34
        AML, DEK::NUP214      1.000     0.769     0.870        13
             AML, ETV6-r      0.667     0.667     0.667         3
           AML, FUS::ERG      1.000     1.000     1.000         2
            AML, KAT6A-r      1.000     0.750     0.857         4
            AML, KMT2A-r      0.866     0.968     0.914       154
            AML, NPM1alt      0.903     0.824     0.862        68
            AML, NUP98-r      0.889     0.828     0.857        29
     AML, RUNX1::RUNX1T1      0.991     0.981     0.986       108
        B-ALL, BCR::ABL1      1.000     0.595     0.746        42
   B-ALL, BCR::ABL

In [11]:
# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))



Top 20 genes driving classification (final model):
       Gene  Importance
0     NUTM1    0.020011
1   RUNX1T1    0.017367
2     APELA    0.015147
4    MEGF10    0.012731
15    MEIS1    0.011593
16     CPA4    0.011399
3     SALL4    0.011272
20      PDC    0.009507
5     KCNN1    0.009444
6     HOXB6    0.009106
11     PBX1    0.008977
9     NDST3    0.008950
78    AGAP1    0.008334
22   KLHL14    0.008211
41    TTC28    0.008194
31  IGF2BP1    0.008121
79    ATXN1    0.008104
10    HOXB5    0.007809
13   ADGRF1    0.007635
57    ASB13    0.007534


In [14]:
# Print all gene names separated by newline
print("\nAll selected genes (final model):")
for gene in gene_importances["Gene"]:
    print(gene)


All selected genes (final model):
NUTM1
RUNX1T1
APELA
MEGF10
MEIS1
CPA4
SALL4
PDC
KCNN1
HOXB6
PBX1
NDST3
AGAP1
KLHL14
TTC28
IGF2BP1
ATXN1
HOXB5
ADGRF1
ASB13
WNT9A
FRRS1
PTPRZ1
PCDH17
GREB1
ANKS1B
PLAAT5
TFPI
DGKI
TLCD5
KRT17
HDAC9
HOXA10
WNT16
SLC22A31
ISM1
HOXA9
PPM1F
BCL6B
SKIDA1
HLF
CHST6
GRIK5
CYP7B1
LOXHD1
KIF16B
DAPK1
HLCS
EVPL
TTC3
SLC25A4
MTMR11
PRDM16
PAK5
CCL17
CLIP2
NUDT11
TUNAR
MSLN
LAMP5
EFEMP1
ABL1
EPN2
EYA4
SALL1
RUNX1
SETD4
DLX3
FPGT-TNNI3K
TRHDE
DDIT4L
BEND6
ARHGEF12
UGGT2
KIF12
ATP8A2
CLCF1
INPP5A
NAP1L2
NRP1
ARHGAP4
ATP9A
DTD1
LIPH
EXOC3L2
PTPN14
CAVIN1
SIPA1L2
TRPC4
CHST8
CYP2S1
WNT7B
ADCY9
COL6A5
ACVR1C
PAQR5
UNCX
FOLR1
CACNA1H
ANXA8
TDRP
LPAR1
KRTAP17-1
RYR2
SOWAHD
FADS3
CUX2
ASB9
CLIC5
DACH1
TMEM50B
HOXA7
MYOM2
IQCJ
DONSON
ELOVL4
KCNJ12
CFAP161
FCRLA
KLF15
SESN1
PSD2
PCDH19
KBTBD11
FRMPD1
FNBP1
TTC16
MATK
REPS2
VIPR2
ARHGAP6
EXD2
CLEC9A
NDRG4
GPR32
UNK
CPXM2
NOVA1
IL7
BAIAP3
PEX5L
ZC2HC1A
HOOK1
GPR88
LGSN
CNTNAP5
NXPE3
ZNRF1
ANO7
CRYGD
PPP1R1C
FHIT
CHST11
SPRED1

In [12]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 0 out of 196

Bottom 20 genes by importance (final model):
         Gene  Importance
166     KRT73    0.002824
187    CDC25B    0.002799
58      LMX1B    0.002794
145    CYP3A7    0.002607
190  BAIAP2L1    0.002482
156    COL6A2    0.002476
139       CD7    0.002397
115   UGT2B11    0.002351
173    LINGO2    0.002330
118     C1QL1    0.002300
148    IGSF9B    0.002120
189     RGS11    0.001973
185      ERFL    0.001907
154    ZNF615    0.001854
149    WNT10A    0.001823
179     KRT72    0.001764
133   TMEM179    0.001727
120      MNX1    0.001406
140    POU4F2    0.001373
170   CTTNBP2    0.001345
